# Parent Document Retriever — 작은 청크로 검색, 큰 청크로 반환

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **ParentDocumentRetriever**가 왜 필요한지, 검색 정확도와 컨텍스트 풍부함의 트레이드오프 이해하기
- `parent_splitter`와 `child_splitter`의 역할 구분하기
- `InMemoryStore`로 부모-자식 문서 관계를 저장하는 방법 익히기

## 사전 지식

- 청크(Chunk) 크기가 검색 정확도와 컨텍스트에 어떻게 영향을 미치는지 이해
- VectorStoreRetriever 기본 사용법

---

```mermaid
flowchart TD
    D[원본 문서]:::input --> PS[parent_splitter<br/>큰 청크 생성<br/>1500자]:::process
    PS --> PD[(부모 문서<br/>InMemoryStore)]:::storage
    D --> CS[child_splitter<br/>작은 청크 생성<br/>300자]:::process
    CS --> VD[(자식 벡터<br/>VectorStore)]:::storage
    Q[사용자 쿼리]:::input --> VD
    VD -- 자식으로 검색 --> R[부모 문서 반환<br/>풍부한 컨텍스트]:::output
    PD --> R

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
    classDef storage fill:#e2d5f1,stroke:#6f42c1,color:#3d1f6e
```

## 핵심 아이디어

| 상황 | 문제 |
|------|------|
| 작은 청크만 사용 | 검색은 정확하지만 컨텍스트 부족 |
| 큰 청크만 사용 | 컨텍스트는 풍부하지만 검색 정확도 낮음 |
| **ParentDocument** | **작은 청크로 검색 + 부모 문서 반환** |

> **실무 팁**: 자식 청크 크기는 300~500자, 부모 청크 크기는 1000~2000자가 일반적으로 좋은 출발점이에요.

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
# ---------------------------------------------------
# 부모-자식 청크 구조로 ParentDocumentRetriever 생성
# ---------------------------------------------------

# ============================================================
# TODO: child_splitter(300자)와 parent_splitter(1500자)를 설정하고 ParentDocumentRetriever를 생성하세요
# 힌트: ParentDocumentRetriever(vectorstore, docstore, child_splitter, parent_splitter)
# 예상 결과: 검색은 300자 자식으로, 반환은 1500자 부모로 수행
# ============================================================

from langchain.retrievers import ParentDocumentRetriever
from langchain_core.stores import InMemoryStore
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader

# 문서 로드
loader = TextLoader("./data/ai-story.txt")
documents = loader.load()

# 두 가지 splitter 설정
# parent_splitter: 반환할 큰 청크 (컨텍스트 풍부)
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
# child_splitter: 검색에 사용할 작은 청크 (정확도 높음)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)

# Vectorstore 및 Docstore
# vectorstore: 자식 청크 임베딩 저장 (검색용)
# InMemoryStore: 부모 문서 원문 저장 (반환용)
vectorstore = FAISS.from_texts(["init"], OpenAIEmbeddings(model="text-embedding-3-small"))
store = InMemoryStore()

# ParentDocumentRetriever 생성


# 문서 추가
parent_retriever.add_documents(documents)

print("✅ ParentDocumentRetriever 생성 완료")
print(f"  - 부모 청크 크기: 1500자")
print(f"  - 자식 청크 크기: 300자 (검색용)")

In [ ]:
# ---------------------------------------------------
# ParentDocumentRetriever 검색 실행 및 반환 문서 크기 확인
# ---------------------------------------------------

# ============================================================
# TODO: parent_retriever.invoke(query)로 검색하고 반환된 문서 길이를 확인하세요
# 힌트: 반환 문서는 자식(300자)이 아닌 부모(1500자) 크기여야 함
# 예상 결과: 각 문서 길이가 1500자 내외로 출력
# ============================================================

# 검색
query = "Word2Vec의 작동 원리는?"


print(f"📝 쿼리: {query}\n")
print("="*80)
print("[ParentDocumentRetriever 결과]")
print("="*80)
for i, doc in enumerate(docs, 1):
    print(f"[문서 {i}]")
    print(f"길이: {len(doc.page_content)}자 ← 부모 문서 (큰 청크)")
    print(f"내용: {doc.page_content[:200]}...")
    print("-"*80)

print("\n💡 장점:")
print("  - 작은 청크로 검색 → 정확도 높음")
print("  - 부모 문서 반환 → 컨텍스트 풍부")
print("  - 최상의 균형 달성")

---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | 작은 청크로 검색(정밀도), 부모 문서로 반환(컨텍스트) |
| 필수 구성 요소 | `child_splitter`, `parent_splitter` (선택), `InMemoryStore`, VectorStore |
| 두 가지 모드 | 전체 문서 반환 / 큰 청크 반환 (`parent_splitter` 유무로 결정) |
| 저장소 역할 | VectorStore — 자식 청크 임베딩 / InMemoryStore — 부모 문서 원문 |
| 성능 트레이드오프 | 인덱싱 시간·메모리 증가, 검색 품질 향상 |

### 파라미터 선택 가이드

| 상황 | 권장 설정 |
|------|-----------|
| 문서가 길고 정밀 검색이 필요할 때 | `child_chunk_size=200`, `parent_chunk_size=2000` |
| 문서가 짧고 전체 문맥이 중요할 때 | `parent_splitter=None` (전체 문서 반환 모드) |
| 메모리가 제한될 때 | `InMemoryStore` 대신 `LocalFileStore` 사용 |

### 다음 노트북 예고

**06-MultiQueryRetriever** — 단일 쿼리 하나로는 놓칠 수 있는 문서를, LLM이 여러 개의 쿼리를 자동 생성해 다각도로 검색하는 방법을 배워요.